In [1]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ════════════════════════════════════════════════════════════════
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.stores import InMemoryStore  
from models import get_model, get_embeddings
from pdf_chunk import text_spliter

llm             = get_model()
embedding_model = get_embeddings()

e:\Rohanta_AI_workbook\adavance_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2009.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Build 3-level hierarchy from your documents
# ════════════════════════════════════════════════════════════════
def build_hierarchy(documents, sizes=(2048, 512, 128)):
    """
    Build parent → mid → leaf hierarchy.
    Each leaf stores its parent_id in metadata for merging.
    Returns:
        leaf_docs   — go into vector store (embedded)
        all_chunks  — go into docstore (id → Document)
    """
    root_splitter = RecursiveCharacterTextSplitter(
        chunk_size=sizes[0], chunk_overlap=50
    )
    mid_splitter  = RecursiveCharacterTextSplitter(
        chunk_size=sizes[1], chunk_overlap=30
    )
    leaf_splitter = RecursiveCharacterTextSplitter(
        chunk_size=sizes[2], chunk_overlap=10
    )

    all_chunks = {}   # id → Document (the docstore)
    leaf_docs  = []   # what gets embedded

    for doc_i, doc in enumerate(documents):

        # Level 1: root chunks
        roots = root_splitter.split_documents([doc])
        for r_i, root in enumerate(roots):
            root_id = f"root_{doc_i}_{r_i}"
            root.metadata["chunk_id"]    = root_id
            root.metadata["level"]       = "root"
            root.metadata["parent_id"]   = None
            all_chunks[root_id] = root

            # Level 2: mid chunks from each root
            mids = mid_splitter.split_documents([root])
            for m_i, mid in enumerate(mids):
                mid_id = f"mid_{doc_i}_{r_i}_{m_i}"
                mid.metadata["chunk_id"]  = mid_id
                mid.metadata["level"]     = "mid"
                mid.metadata["parent_id"] = root_id
                all_chunks[mid_id] = mid

                # Level 3: leaf chunks from each mid
                leaves = leaf_splitter.split_documents([mid])
                for l_i, leaf in enumerate(leaves):
                    leaf_id = f"leaf_{doc_i}_{r_i}_{m_i}_{l_i}"
                    leaf.metadata["chunk_id"]  = leaf_id
                    leaf.metadata["level"]     = "leaf"
                    leaf.metadata["parent_id"] = mid_id
                    all_chunks[leaf_id] = leaf
                    leaf_docs.append(leaf)

    return leaf_docs, all_chunks


leaf_docs, all_chunks = build_hierarchy(text_spliter, sizes=(2048, 512, 128))

print(f"Total chunks in hierarchy : {len(all_chunks)}")
print(f"Leaf docs (embedded)      : {len(leaf_docs)}")
print(f"  — roots : {sum(1 for c in all_chunks.values() if c.metadata['level']=='root')}")
print(f"  — mids  : {sum(1 for c in all_chunks.values() if c.metadata['level']=='mid')}")
print(f"  — leaves: {sum(1 for c in all_chunks.values() if c.metadata['level']=='leaf')}")

Total chunks in hierarchy : 259
Leaf docs (embedded)      : 161
  — roots : 49
  — mids  : 49
  — leaves: 161


In [3]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Index only leaf docs into Chroma
# ════════════════════════════════════════════════════════════════
vectorstore = Chroma.from_documents(
    leaf_docs,
    embedding_model,
    collection_name="auto_merge_leaves"
)

print("Leaf docs indexed into Chroma ✓")

Leaf docs indexed into Chroma ✓


In [4]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Auto Merging logic
# ════════════════════════════════════════════════════════════════
from collections import defaultdict

def auto_merge_retriever(query, k=10, merge_threshold=0.5):
    """
    1. Retrieve top-k leaf chunks
    2. Group by parent_id
    3. If matched_leaves / total_leaves >= threshold → promote to parent
    4. Return merged or individual docs
    """

    # Step 1: retrieve matched leaves
    matched_leaves = vectorstore.similarity_search(query, k=k)

    # Step 2: group matched leaves by their parent
    parent_to_matched = defaultdict(list)
    for leaf in matched_leaves:
        parent_id = leaf.metadata.get("parent_id")
        if parent_id:
            parent_to_matched[parent_id].append(leaf)

    # Step 3: merge decision per parent
    result_docs = []
    already_added = set()

    for parent_id, matched in parent_to_matched.items():
        parent_doc = all_chunks.get(parent_id)
        if parent_doc is None:
            result_docs.extend(matched)
            continue

        # Count total children of this parent
        total_children = sum(
            1 for c in all_chunks.values()
            if c.metadata.get("parent_id") == parent_id
        )

        match_ratio = len(matched) / max(total_children, 1)

        if match_ratio >= merge_threshold:
            # Promote: return parent instead of individual leaves
            if parent_id not in already_added:
                result_docs.append(parent_doc)
                already_added.add(parent_id)
            print(f"MERGED  → {parent_id} "
                  f"({len(matched)}/{total_children} = {match_ratio:.0%})")
        else:
            # Keep individual leaves
            for leaf in matched:
                lid = leaf.metadata["chunk_id"]
                if lid not in already_added:
                    result_docs.append(leaf)
                    already_added.add(lid)
            print(f"KEPT    → {len(matched)} leaves from {parent_id} "
                  f"({match_ratio:.0%} < threshold)")

    return result_docs


def pretty_print_docs(docs):
    print(f"\n{'-'*80}\n".join(
        [f"Doc {i+1} [{d.metadata.get('level','?')} | "
         f"{len(d.page_content)} chars]:\n\n{d.page_content}"
         for i, d in enumerate(docs)]
    ))

In [5]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Test retrieval
# ════════════════════════════════════════════════════════════════
query = "what is Rohanta's current university and what does he study?"
docs  = auto_merge_retriever(query, k=10, merge_threshold=0.5)
print(f"\nFinal docs returned: {len(docs)}\n")
pretty_print_docs(docs)

KEPT    → 1 leaves from mid_14_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_6_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_47_0_0 (20% < threshold)
KEPT    → 2 leaves from mid_0_0_0 (40% < threshold)
KEPT    → 1 leaves from mid_41_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_1_0_0 (20% < threshold)
MERGED  → mid_13_0_0 (1/1 = 100%)
KEPT    → 1 leaves from mid_16_0_0 (17% < threshold)
KEPT    → 1 leaves from mid_17_0_0 (17% < threshold)

Final docs returned: 10

Doc 1 [leaf | 128 chars]:

Before transitioning fully into industry, Rohanta taught undergraduate Computer Science to 100+ students. He designed curriculum
--------------------------------------------------------------------------------
Doc 2 [leaf | 116 chars]:

Rohanta is currently working as a self-employed AI/ML Consultant while simultaneously pursuing his MSc in Artificial
--------------------------------------------------------------------------------
Doc 3 [leaf | 117 chars]:

Rohanta is looking for full-ti

In [7]:
# ════════════════════════════════════════════════════════════════
# CELL 6 — Full LCEL chain with Auto Merging
# ════════════════════════════════════════════════════════════════
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.
If the answer is not in the context, say "I don't know".

Context: {context}
Question: {question}
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

chain = (
    {
        "context": RunnableLambda(
            lambda q: format_docs(
                auto_merge_retriever(q, k=10, merge_threshold=0.5)
            )
        ),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("what is Rohanta's current university?"))

KEPT    → 1 leaves from mid_14_0_0 (20% < threshold)
MERGED  → mid_0_0_0 (3/5 = 60%)
KEPT    → 1 leaves from mid_6_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_47_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_1_0_0 (20% < threshold)
KEPT    → 1 leaves from mid_41_0_0 (20% < threshold)
KEPT    → 2 leaves from mid_17_0_0 (33% < threshold)
I don't know.
